# W2 Homework — Prompting Techniques (graded)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week02/W2_hw_prompting.ipynb)

**Goal.** Apply the four techniques of the W2 lab to new tasks: chain-of-thought by instruction, format pinned by a worked example, self-consistency written by you, and chain-of-thought by worked examples.

Sections: setup (1) → assignment 1, instruction (2) → assignment 2, exemplar (3) → assignment 3, the vote (4) → assignment 4, worked solutions (5) → completion and submission (6).

Each assignment has one block marked `### FILL IN (START) ###` … `### FILL IN (END) ###` and a fixed check below it. Only the block changes. The check prints, for each test item, whether it passed, then one summary line. Grading reads the saved outputs, so run the notebook top to bottom before submitting.

**This homework is collected.** Fill in Section 1.3, make every check pass, and follow Section 6 to submit to the LMS. Due: before the W3 class.


## 1. Setup


### 1.1 Installation

`aisuite` runs the same code on OpenAI or Anthropic keys.

*Do:* run the cell (about 30 seconds, once per session).


In [ ]:
%pip install -q "aisuite[openai,anthropic]"

### 1.2 API key and model

**API key** = the secret string that identifies your account and bills usage to it (issuing steps: the API Setup guide on the course site). Do not share the notebook with the key inside.

*Do:* replace `PASTE-YOUR-KEY-HERE` with your key and run the cell. Nothing prints.


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"   # Anthropic accounts: MODEL = "anthropic:claude-haiku-4-5" and set ANTHROPIC_API_KEY instead

### 1.3 Submission identity

The completion check in Section 6 prints these values; a blank name fails it.

*Do:* fill in your name and student ID, run the cell.


In [ ]:
STUDENT_NAME = ""
STUDENT_ID = ""

print("submitting as:", STUDENT_NAME, STUDENT_ID)


### 1.4 Client and a first call

The same call that opened W1. Every later cell repeats this shape; only the prompt string changes.

*Do:* run the cell. The output must be exactly `ready`.


In [ ]:
import aisuite

client = aisuite.Client()

messages = [{"role": "user", "content": "Reply with exactly: ready"}]
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
response.choices[0].message.content

Any error here is a setup problem: recheck the API Setup guide before continuing.


## 2. Assignment 1 — An instruction that elicits the solution ✍️

`COT_INSTRUCTION` is appended after each word problem. The starter is empty, so the extraction finds no `ANSWER:` line. Both problems must pass.

*Do:* write the instruction, run, iterate until both lines read `PASS`.

Hints: demand the solution step by step and the exact final line `ANSWER: <number>`.


In [ ]:
import re

WORD_PROBLEMS = [
    ("A library has 4 shelves with 38 books each. During the day 47 books are checked out and 26 are returned. How many books are on the shelves now?", "131"),
    ("A bakery bakes 12 trays of 24 muffins. 6 trays are sold whole and 37 more muffins are sold singly. How many muffins remain?", "107"),
]

### FILL IN (START) ###
# Write the instruction appended after each problem: demand the solution step by step
# and the exact last line ANSWER: <number>
COT_INSTRUCTION = ""
### FILL IN (END) ###

a1_passed = 0
for question, expected in WORD_PROBLEMS:
    messages = [
        {"role": "user", "content": question + "\n" + COT_INSTRUCTION},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    match = re.search(r"ANSWER:\s*(\d+)", output)
    extracted = None
    if match:
        extracted = match.group(1)
    correct = extracted == expected
    a1_passed += correct
    print(correct, "extracted", extracted, "expected", expected, "| output ended:", output.strip()[-60:])

assignment_1 = a1_passed == len(WORD_PROBLEMS)
print("assignment 1:", a1_passed, "/", len(WORD_PROBLEMS))


## 3. Assignment 2 — An exemplar that pins the format ✍️

The check accepts exactly one output shape per note: `DATE: 2026-03-10`, nothing before or after. Instructions leave something loose; a worked exemplar shows the shape. Both notes must pass.

*Do:* add one or two worked note → answer exemplars inside `DATE_PROMPT` above the test note; iterate until both lines read `PASS`.

Hints: format each exemplar exactly as the output should look (a note about June 9th, 2026 answered `DATE: 2026-06-09` on its own line); keep the test note last. Exemplar notes must not be the test notes.


In [ ]:
TEST_NOTES = [
    ("Team offsite moved from March 3rd to March 10th, 2026.", "DATE: 2026-03-10"),
    ("Deadline for the grant report extended from April 1st to April 22nd, 2026.", "DATE: 2026-04-22"),
]

### FILL IN (START) ###
# Insert one or two worked exemplars between the first line and the final Note line,
# each as:  Note: <a note with two dates>  /  DATE: YYYY-MM-DD
DATE_PROMPT = """Extract the final date from the note.

Note: {note}"""
### FILL IN (END) ###

a2_passed = 0
for note, expected in TEST_NOTES:
    messages = [
        {"role": "user", "content": DATE_PROMPT.format(note=note)},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    correct = output.strip() == expected
    a2_passed += correct
    print(correct, "got", output.strip(), "| expected", expected)

assignment_2 = a2_passed == len(TEST_NOTES)
print("assignment 2:", a2_passed, "/", len(TEST_NOTES))


## 4. Assignment 3 — Self-consistency, written by you ✍️

A statement the direct prompt tends to miss: a spider that lost three legs, golden answer 5. Two things are yours: the chain-of-thought prompt and the sampling loop.

*Do:* (1) write `COT_PROMPT`, a template with the `{statement}` placeholder that makes the model reason inside `<thinking>` tags and put the integer alone inside `<answer>` tags, the prompt shape of lab Section 4.5; (2) write the loop that calls it five times at `temperature=1.0` and appends the text between the `<answer>` tags of each reply to `samples`. The vote below is given. Iterate until the line reads `PASS`.

Hints: the loop is the cell of lab Section 6.2 with the statement replaced. Keep `temperature=1.0`; at 0.0 the five samples are five copies of one path. The check requires at least five samples and a majority equal to the golden answer.


In [ ]:
from collections import Counter

TRICKY = {"animal_statement": "The animal is a spider that lost three legs.", "golden_answer": "5"}

### FILL IN (START) ###
# 1. COT_PROMPT: keep the {statement} placeholder; replace the last line with two instructions:
#    reason step by step inside <thinking> tags, then put the final answer, the integer alone,
#    inside <answer> tags.
COT_PROMPT = """You will be provided a statement about an animal and your job is to determine how many legs that animal has.

Here is the animal statement.
<animal_statement>{statement}</animal_statement>

How many legs does the animal have?"""

# 2. The sampling loop: five calls of COT_PROMPT on TRICKY["animal_statement"] at temperature=1.0;
#    append the text between the <answer> tags of each reply to samples.
samples = []
### FILL IN (END) ###

votes = Counter(samples)
majority = None
if votes:
    majority = votes.most_common(1)[0][0]
assignment_3 = len(samples) >= 5 and majority == TRICKY["golden_answer"]

print("samples:", samples)
print("majority:", majority, "  golden:", TRICKY["golden_answer"], "  samples:", len(samples))
if assignment_3:
    print("PASS  assignment 3")
else:
    print("FAIL  assignment 3")


## 5. Assignment 4 — Worked examples that elicit the solution ✍️

Six arithmetic expressions in Roman numerals. The check requires the right number and a **last line** of exactly `ANSWER: <number>`. The starter gives the instruction alone, and the last-line check tends to fail. Target: **at least 5 of 6**; no evalset expression may appear in the prompt (the check reports a leak).

*Do:* write `ROMAN_PROMPT` as few-shot CoT: at least two worked examples, each converting the numerals, computing, and closing with `ANSWER: <number>` on its own line, then the test expression laid out the same way. Iterate until `PASS`.

Hints: choose exemplars not in `ROMAN_EVALSET` (for instance `XII + VII`); write each solution the way the model's should look; keep `{expression}` last.


In [ ]:
import re

ROMAN_EVALSET = [
    ("XIV + IX", "23"),
    ("XLII - XXIX", "13"),
    ("CD + XC", "490"),
    ("MMXXVI - MCMXCIV", "32"),
    ("XCIX + I", "100"),
    ("LXXX / XVI", "5"),
]

### FILL IN (START) ###
# Insert at least two worked examples between the first line and the {expression} line.
# Each example: the expression, the numerals converted, the computation, then a line ANSWER: <number>
# Do not use any expression from ROMAN_EVALSET.
ROMAN_PROMPT = """Evaluate the expression written in Roman numerals. Give the result as ANSWER: <number>.

{expression}"""
### FILL IN (END) ###

leaked = []
for expression, expected in ROMAN_EVALSET:
    if expression in ROMAN_PROMPT:
        leaked.append(expression)
n_exemplars = len(re.findall(r"^ANSWER:\s*\d+\s*$", ROMAN_PROMPT, re.M))   # answer lines of the worked examples

roman_score = 0
for expression, expected in ROMAN_EVALSET:
    messages = [
        {"role": "user", "content": ROMAN_PROMPT.format(expression=expression)},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.0)
    output = response.choices[0].message.content
    last_line = output.strip().splitlines()[-1].strip()
    correct = last_line == "ANSWER: " + expected
    roman_score += correct
    print(correct, expression, "| last line:", last_line)

assignment_4 = roman_score >= 5 and n_exemplars >= 2 and not leaked
print("assignment 4:", roman_score, "/ 6   exemplars:", n_exemplars, "  leaked:", leaked)
if assignment_4:
    print("PASS  assignment 4")
else:
    print("FAIL  assignment 4")


## 6. Completion and Submission

Completion: the identity line is filled and all four assignments pass. Grading checks these facts from the saved outputs and compares each fill-in block with the starter; it never grades prose.

*Do:* run the notebook top to bottom once more so every output is saved, confirm every row below reads `PASS`, download the notebook (**File → Download → Download .ipynb**), and upload that file to the LMS assignment. Keep the file name and the check cells as they are.


In [ ]:
completion = {
    "name and student ID filled in (1.3)":            bool(STUDENT_NAME.strip()) and bool(STUDENT_ID.strip()),
    "assignment 1: instruction elicits ANSWER on both problems": assignment_1,
    "assignment 2: exemplar pins the format on both notes":      assignment_2,
    "assignment 3: the vote finds the answer":                   assignment_3,
    "assignment 4: worked examples reach >= 5/6":                assignment_4,
}

print("submitted by:", STUDENT_NAME, STUDENT_ID)
print()
for item, ok in completion.items():
    if ok:
        print("PASS ", item)
    else:
        print("FAIL ", item)

if all(completion.values()):
    print("\nHOMEWORK COMPLETE — download and submit to the LMS")
else:
    print("\nNOT COMPLETE YET")
